# Notebook de travail

Projet : Smart City Energy Forecasting — Tetouan


Importations

In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

#chargement des données horaires générées précédemment
df = pd.read_csv("../data/raw/Tetuan City power consumption.csv")
df = df.rename(columns={
    "DateTime": "datetime", "Temperature": "temperature", "Humidity": "humidity",
    "Wind Speed": "wind_speed", "general diffuse flows": "general_diffuse_flows",
    "diffuse flows": "diffuse_flows", "Zone 1 Power Consumption": "zone1_power",
    "Zone 2  Power Consumption": "zone2_power", "Zone 3  Power Consumption": "zone3_power"
    })
df["datetime"] = pd.to_datetime(df["datetime"])
df = df.set_index("datetime").sort_index()
df_hourly = df.resample("1h").mean()

print(f"Dimensions intiales :{df_hourly.shape}")

Dimensions intiales :(8736, 8)


Gestion des valeurs manquantes (Sécurité)

In [12]:
def handle_missing_values(data, limit=3):
    # Interpolation temporelle pour les petits trous
    data_clean = data.interpolate(method='time', limit=limit)
    # Remplissage avant/arrière pour les restes éventuels
    data_clean = data_clean.ffill(limit=limit).bfill(limit=limit)
    return data_clean

df_clean = handle_missing_values(df_hourly)
print(f"Valeurs manquantes restantes : {df_clean.isna().sum().sum()}")

Valeurs manquantes restantes : 0


Détection d'anomalies par Z-Score Glissant (Rolling Z-Score)

In [13]:
# On cherche les erreurs techniques, pas les pics de consomation légitimes
# Cellule 3 : Détection d'anomalies (Z-Score Glissant) et Sauvegarde
def detect_rolling_outliers(series, window=24, threshold=3.5):
    rolling_mean = series.rolling(window=window, center=True, min_periods=3).mean()
    rolling_std = series.rolling(window=window, center=True, min_periods=3).std()
    zscore = ((series - rolling_mean) / rolling_std.replace(0, np.nan)).abs()
    return zscore > threshold

outliers_temp = detect_rolling_outliers(df_clean['temperature'])
print(f"Anomalies détectées sur la température : {outliers_temp.sum()}")

df_clean["target"] = df_clean["zone1_power"]
df_clean["total_load"] = df_clean["zone1_power"] + df_clean["zone2_power"] + df_clean["zone3_power"]

df_clean.to_csv("../data/processed/tetouan_hourly_clean.csv", index=True)
print("Prétraitement terminé et données sauvegardées avec succès.")

Anomalies détectées sur la température : 0
Prétraitement terminé et données sauvegardées avec succès.
